# 10 · Model Context Protocol (MCP)

**Goal:** package a set of tools, data and instructions **once**, as an MCP server, so any MCP-aware app can use them. Then build the client side and let Claude use the server's tools.

| | |
|---|---|
| **You will learn** | What MCP is and isn't, how a client and server talk, defining tools, testing a server on its own, writing a client, resources, prompts, and a small chat app that uses all three |
| **Prerequisites** | [Notebook 05](05_tool_use.ipynb), for the tool loop |
| **Run time** | About 2 minutes |
| **Cost** | Under $0.10 expected. This notebook stops itself at $0.20 |
| **Paid cells** | Chapters 3, 6, 8 and 10. Chapters 4, 5, 7 and 9 are free |

If you haven't yet, set a spend limit in the [Claude Console](https://platform.claude.com/settings/billing) (**Billing**, then **Spend limits**).

**The story.** Halvora's annual report (`data/halvora_report.md`, the fictional company used across these notebooks) lives in one place. Every new app that wants to read it, whether a chatbot, a dashboard or Claude Code, would need its own integration. Instead, we wrap the documents in an **MCP server** once, so any MCP client can plug in.

**Files.** The server is a small Python package, `halvora_mcp/`, that grows by one file per chapter. The notebook writes each file with `%%writefile`, so you can read every line of the server here. The client lives in the notebook.

> **Different from the course:** the course's code uses version 1 of the MCP Python SDK. This notebook uses version 2.3, where four things changed:
>
> | Course (SDK 1.x) | Today (SDK 2.x) |
> |---|---|
> | `from mcp.server.fastmcp import FastMCP` | `from mcp.server.mcpserver import MCPServer` |
> | camelCase fields: `inputSchema`, `isError`, `mimeType`, `uriTemplate` | snake_case: `input_schema`, `is_error`, `mime_type`, `uri_template` |
> | `session.read_resource(AnyUrl(uri))` | `session.read_resource(uri)`, with a plain string |
> | `raise ValueError(...)` reaches Claude | Only `ToolError` and `ResourceError` messages reach Claude. Any other exception becomes a generic error |
>
> The ideas, the decorators (`@mcp.tool`, `@mcp.resource`, `@mcp.prompt`) and the client methods are the same.

## Setup

The same setup as the earlier notebooks, plus the tool-loop helpers from notebook 05. Run the cells from top to bottom.

In [1]:
import os
import sys

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(override=True)  # reads your keys from the .env file

api_key = os.getenv("ANTHROPIC_API_KEY", "")
assert api_key and api_key != "your_key_here", "Add your API key to .env, restart the kernel, and run this again."

client = Anthropic()  # finds ANTHROPIC_API_KEY by itself
MODEL = "claude-sonnet-5-5"

print("Ready. Using model:", MODEL)

Ready. Using model: claude-sonnet-5-5


In [2]:
INPUT_PRICE = 2.00    # dollars per 1 million input tokens (Sonnet 5.5)
OUTPUT_PRICE = 10.00  # dollars per 1 million output tokens (Sonnet 5.5)
BUDGET_USD = 0.20     # the most this notebook should spend

spent = 0.0


def track(response):
    global spent
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    cost = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000
    spent += cost
    print(f"[input={input_tokens:,} output={output_tokens:,} cost=${cost:.5f} | spent ${spent:.4f} of ${BUDGET_USD:.2f}]")
    if spent > BUDGET_USD:
        raise RuntimeError(f"Budget reached: spent ${spent:.4f}. Raise BUDGET_USD if you mean to spend more.")
    return response


def text_from_message(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""

## Chapter 1 · Introducing MCP

**Analogy: UPI.** Your payment app doesn't build a separate connection to every bank. Each bank joins UPI **once**, and every app can pay through it. MCP does the same for AI apps: a service packages its tools **once**, as an **MCP server**, and any app with an **MCP client** can use them.

```
 Your app (MCP client)  ──MCP──▶  GitHub MCP server   ──▶  GitHub API
                        ──MCP──▶  Halvora docs server ──▶  the documents
```

An MCP server offers three kinds of thing, called **primitives**:

| Primitive | What it is | Who decides to use it | Chapter |
|---|---|---|---|
| **Tools** | Actions, such as read or edit | **Claude** | 4 |
| **Resources** | Data the app can put into the prompt | **The app**, for example on an `@mention` | 7 |
| **Prompts** | Ready-made, tested instructions | **The user**, for example with a `/command` | 9 |

**Isn't MCP just tool use?** No, and this is the most common mix-up.

- **Tool use** (notebook 05) is *how* Claude asks for a tool and gets a result.
- **MCP** is *who builds and runs* the tools. In notebook 05 you wrote every schema and function yourself. With MCP the server already has them, and your app just lists them and passes calls along.

They work together: in Chapter 6 Claude uses ordinary tool use, but the tools come from an MCP server.

## Chapter 2 · MCP clients: how the two sides talk

The **MCP client** connects your app to an MCP server and passes messages between them, so your code can simply call `list_tools()` or `call_tool(...)`.

**MCP doesn't care how messages travel:**

- **stdio** (standard input and output): the client **starts the server as a child process** on the same machine and talks to it through pipes. We use this. It's the usual setup for a local server.
- **Streamable HTTP**: for servers running somewhere else on the network.

**Message types** (defined by the MCP spec, and built for you by the SDK):

| Client sends | Server replies | Meaning |
|---|---|---|
| `initialize` | capabilities | "Hello, what can you do?" |
| `ListToolsRequest` | `ListToolsResult` | "What tools do you have?" |
| `CallToolRequest` | `CallToolResult` | "Run this tool with these arguments." |
| `ListResourcesRequest` / `ReadResourceRequest` | results | Chapters 7 and 8 |
| `ListPromptsRequest` / `GetPromptRequest` | results | Chapters 9 and 10 |

**The full flow** for *"What does the cybersecurity document say?"*:

```
1. user ──question──▶ your app
2. your app ──ListToolsRequest──▶ MCP server        (get the tool list)
3. your app ──question + tools──▶ Claude
4. Claude ──tool_use: read_doc_contents──▶ your app
5. your app ──CallToolRequest──▶ MCP server ──▶ the document
6. MCP server ──CallToolResult──▶ your app
7. your app ──tool_result──▶ Claude
8. Claude ──final answer──▶ your app ──▶ user
```

Steps 3, 4, 7 and 8 are the tool loop from notebook 05. MCP only changes **where the tools come from**: steps 2, 5 and 6. You run exactly this in Chapter 6.

## Chapter 3 · Project setup (paid)

**What we're building:** a document server for Halvora's report. The report is split into one document per section (`software-engineering`, `cybersecurity-analysis`, and so on), kept **in memory**, with no database.

> **In real projects you build either a client or a server.** You build a server to offer your service to others, or a client to connect to existing servers. We build both here only to see how they fit together.

**A habit worth copying:** before adding each layer, ask the running system **one question whose answer you already know**.

The first cell creates the folders and checks the MCP SDK is installed. `SERVER_PARAMS` tells a client how to start our server: run `halvora_mcp/server.py` with this notebook's Python.

In [3]:
from importlib.metadata import version
from pathlib import Path

from mcp import StdioServerParameters

assert sys.version_info >= (3, 10), "The MCP SDK needs Python 3.10 or newer."

SERVER_DIR = Path("halvora_mcp")
SERVER_DIR.mkdir(exist_ok=True)

OUTPUT_DIR = Path("outputs/10_mcp")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SERVER_LOG = open(OUTPUT_DIR / "server.log", "a")   # the server's error output goes here, not into Jupyter

SERVER_PARAMS = StdioServerParameters(
    command=sys.executable,                  # this notebook's Python, from the .venv
    args=[str(SERVER_DIR / "server.py")],
)

print(f"Python {sys.version.split()[0]} | mcp {version('mcp')} | anthropic {version('anthropic')}")

Python 3.12.13 | mcp 2.3.0 | anthropic 2.5.0


The next two cells write the first two server files. `%%writefile` is a Jupyter "magic" that **saves** a cell's text to a file instead of running it.

- `core.py` creates the server object and loads the documents.
- `server.py` is the program a client starts. It imports `core`, then any chapter files that exist so far, then runs.

In [4]:
%%writefile halvora_mcp/core.py
# core.py: the server object and the documents it manages.
import re
from pathlib import Path

from mcp.server.mcpserver import MCPServer          # called FastMCP in SDK 1.x

mcp = MCPServer("HalvoraDocs", log_level="ERROR")   # the name clients see; log only real errors


def _load_docs():
    """Split the report into one document per '## ' section, keyed by a short id."""
    report = (Path(__file__).parent.parent / "data" / "halvora_report.md").read_text(encoding="utf-8")
    docs = {}
    for section in re.split(r"\n(?=## )", report)[1:]:              # [1:] skips the title block
        title = section.splitlines()[0].lstrip("# ").strip()        # "Section 2: Software Engineering"
        title = title.split(":", 1)[-1].strip()                     # "Software Engineering"
        doc_id = re.sub(r"[^a-z0-9]+", "-", title.lower()).strip("-")   # "software-engineering"
        docs[doc_id] = section.strip()
    return docs


docs = _load_docs()                    # doc_id -> text, kept in memory

Writing halvora_mcp/core.py


In [5]:
%%writefile halvora_mcp/server.py
# server.py: the program an MCP client starts. It loads whichever chapter files exist.
import importlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))     # so "import core" works from anywhere
from core import mcp

for part in ["tools", "resources", "prompts"]:      # chapters 4, 7 and 9
    if (Path(__file__).parent / f"{part}.py").exists():
        importlib.import_module(part)

if __name__ == "__main__":
    mcp.run()                                        # the default transport is stdio

Writing halvora_mcp/server.py


**The baseline check:** Claude **without** any MCP server. One question we can verify, and one it can't possibly answer yet.

In [6]:
sys.path.insert(0, str(SERVER_DIR))
from core import docs                                 # peek at what the server will serve

print(f"{len(docs)} documents: {list(docs)}\n")

answers = {}
for question in ["What's 1+1? Answer with just the number.",
                 "What does Halvora's cybersecurity document say about INC-2023-Q4-011? One sentence."]:
    response = track(client.messages.create(
        model=MODEL,
        max_tokens=300,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": question}],
    ))
    answers[question] = text_from_message(response)
    print(f"Q: {question}\nA: {answers[question]}\n")

assert "2" in answers["What's 1+1? Answer with just the number."]
assert "cybersecurity-analysis" in docs

15 documents: ['executive-summary', 'table-of-contents', 'methodology', 'medical-research', 'software-engineering', 'financial-analysis', 'product-development', 'legal-developments', 'human-resources', 'marketing-and-brand', 'supply-chain-and-operations', 'sustainability', 'cybersecurity-analysis', 'research-partnerships', 'future-outlook']

[input=23 output=3 cost=$0.00008 | spent $0.0001 of $0.20]
Q: What's 1+1? Answer with just the number.
A: 2

[input=41 output=78 cost=$0.00086 | spent $0.0009 of $0.20]
Q: What does Halvora's cybersecurity document say about INC-2023-Q4-011? One sentence.
A: I don't have access to Halvora's cybersecurity document, and no file or text was included in our conversation, so I can't say what it states about INC-2023-Q4-011. If you paste the relevant section, I'll summarize it in one sentence.



**What to notice**

- `1+1` gives `2`, so the API key and the model work. That's a baseline you **checked**, not just "a reply came back".
- On the cybersecurity question, Claude has **no way** to see Halvora's documents, so it should say it doesn't know. That's the gap the server will fill.
- `core.py` turned each `## ` section into a document. The ids, such as `software-engineering` and `cybersecurity-analysis`, are what Claude will pass to the tools.

## Chapter 4 · Defining tools with MCP (free)

In notebook 05 every tool had **two** parts: a JSON schema and a Python function. With the MCP SDK you write **only the function**:

- `@mcp.tool(name=..., description=...)` registers it.
- **Type hints and `Field(description=...)`** become the JSON schema, built for you by Pydantic.

Two tools: **read** a document, and **edit** one by find-and-replace.

In [7]:
%%writefile halvora_mcp/tools.py
# tools.py: ACTIONS Claude can take. (Chapter 4)
from mcp.server.mcpserver.exceptions import ToolError
from pydantic import Field

from core import docs, mcp


@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a Halvora document and return it as a string.",
)
def read_document(
    doc_id: str = Field(description="Id of the document to read, e.g. 'software-engineering'"),
) -> str:
    if doc_id not in docs:
        raise ToolError(f"Doc with id {doc_id} not found. Available: {', '.join(docs)}")
    return docs[doc_id]


@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the document's content with a new string.",
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(description="The text to replace. Must match exactly, including whitespace."),
    new_str: str = Field(description="The new text to insert in place of the old text."),
) -> str:
    if doc_id not in docs:
        raise ToolError(f"Doc with id {doc_id} not found. Available: {', '.join(docs)}")
    if old_str not in docs[doc_id]:
        raise ToolError("old_str was not found in the document. Read it first and copy the text exactly.")
    docs[doc_id] = docs[doc_id].replace(old_str, new_str)
    return f"Edited {doc_id}: replaced {len(old_str)} characters."

Writing halvora_mcp/tools.py


We can ask the server object directly for the schemas it generated, with no client and no API call. (`await` works at the top level here because Jupyter runs an event loop. In a normal `.py` script you'd wrap it in `asyncio.run(...)`.)

In [8]:
import importlib
import json

import core
import tools
importlib.reload(tools)            # picks up your edits if you re-run the cell above

server_tools = await core.mcp.list_tools()
for tool in server_tools:
    print(f"{tool.name}: {tool.description}")
    print(json.dumps(tool.input_schema, indent=2), "\n")

assert {tool.name for tool in server_tools} == {"read_doc_contents", "edit_document"}

read_doc_contents: Read the contents of a Halvora document and return it as a string.
{
  "properties": {
    "doc_id": {
      "description": "Id of the document to read, e.g. 'software-engineering'",
      "title": "Doc Id",
      "type": "string"
    }
  },
  "required": [
    "doc_id"
  ],
  "title": "read_documentArguments",
  "type": "object"
} 

edit_document: Edit a document by replacing a string in the document's content with a new string.
{
  "properties": {
    "doc_id": {
      "description": "Id of the document that will be edited",
      "title": "Doc Id",
      "type": "string"
    },
    "old_str": {
      "description": "The text to replace. Must match exactly, including whitespace.",
      "title": "Old Str",
      "type": "string"
    },
    "new_str": {
      "description": "The new text to insert in place of the old text.",
      "title": "New Str",
      "type": "string"
    }
  },
  "required": [
    "doc_id",
    "old_str",
    "new_str"
  ],
  "title": "edit_do

**What to notice**

- Compare this with notebook 05, where you typed every `"type": "object", "properties": ...` by hand. Here `doc_id: str = Field(description=...)` produced the same structure, including `"required"`.
- **Expected errors use `ToolError`.** An unknown `doc_id` raises a `ToolError` whose message lists the valid ids, and Claude reads that message and can try again.
- **Why not `ValueError`, as the course does?** In SDK 2.x, any exception that **isn't** a `ToolError` is treated as a crash. Claude would only see *"Error executing tool read_doc_contents"*, with the details hidden in the server log, so it would be stuck guessing.
- **Two fixes to the course's code:** `edit_document` now **returns a confirmation** (the course's version returned nothing), and it **errors if `old_str` isn't found** (the course's `str.replace` would silently do nothing, and Claude would think the edit worked).

## Chapter 5 · The server inspector: test the server on its own (free)

The **MCP Inspector** is like Postman for MCP servers: a browser page that connects to your server, so you can list and run its tools, resources and prompts **without** Claude or a full app.

It needs **Node.js** installed. From the `notebooks/` folder, with the virtual environment active, run this in a terminal:

```bash
mcp dev halvora_mcp/server.py
```

It prints a local address. Open it, click **Connect**, go to **Tools**, then **List Tools**, pick `read_doc_contents`, enter `software-engineering`, and click **Run Tool**. The Inspector's look changes between versions, but the steps stay similar.

**In this notebook,** the next cell does the same in code. It starts the server over stdio, opens an MCP session, and runs a **chain check**: read, edit, then read again.

In [9]:
from mcp import ClientSession
from mcp.client.stdio import stdio_client


def show_result(label, result):
    text = " ".join(item.text for item in result.content if item.type == "text")
    status = "error" if result.is_error else "ok"
    print(f"   {label:<16} {status:<6} {text[:100]}")


async with stdio_client(SERVER_PARAMS, errlog=SERVER_LOG) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        init = await session.initialize()                                     # the handshake
        print(f"Connected to '{init.server_info.name}'")
        print("Tools:", [tool.name for tool in (await session.list_tools()).tools], "\n")

        doc = {"doc_id": "human-resources"}
        show_result("read (before)", await session.call_tool("read_doc_contents", doc))
        show_result("edit", await session.call_tool("edit_document",
                    {**doc, "old_str": "Headcount grew", "new_str": "HEADCOUNT GREW"}))
        after = await session.call_tool("read_doc_contents", doc)
        show_result("read (after)", after)
        bad = await session.call_tool("read_doc_contents", {"doc_id": "nope"})
        show_result("bad doc_id", bad)

assert "HEADCOUNT GREW" in after.content[0].text
assert bad.is_error

Connected to 'HalvoraDocs'
Tools: ['read_doc_contents', 'edit_document'] 

   read (before)    ok     ## Section 6: Human Resources

Headcount grew from 1,420 to 1,730 employees. We hired 310 people, mo
   edit             ok     Edited human-resources: replaced 14 characters.
   read (after)     ok     ## Section 6: Human Resources

HEADCOUNT GREW from 1,420 to 1,730 employees. We hired 310 people, mo
   bad doc_id       error  Error executing tool read_doc_contents: Doc with id nope not found. Available: executive-summary, ta


**What to notice**

- **Two `async with` blocks.** `stdio_client` starts the server **process** and returns its two streams. `ClientSession` speaks MCP over those streams. Leaving the blocks closes the session and **stops the server process**.
- `session.initialize()` is the MCP **handshake**. Every session starts with it.
- **The chain check** (read, edit, read) proves the edit really happened, not just that the tool said "ok".
- **A bad `doc_id` comes back with `is_error=True`** and our `ToolError` message. It doesn't crash the client.
- **The edit doesn't last.** `docs` lives in the server **process's** memory, and that process ended when the `async with` closed. Run the cell again, and "before" shows the original text. A real server would save to a database or a file.
- The server's error output goes to `outputs/10_mcp/server.log`. Look there if a connection fails.

## Chapter 6 · Implementing a client (paid)

Chapter 5 worked, but every use needs two nested `async with` blocks. The course wraps that in an **`MCPClient` class**:

- **Our class** is a friendly wrapper with exactly the methods the app needs.
- **`ClientSession`**, from the SDK, is the real connection inside it.
- The class works with `async with MCPClient(...) as mcp_client:`, so cleanup happens automatically, even after an error.

`AsyncExitStack` makes that easy. It collects everything we open (the server process, then the session) and closes it all, in reverse order, when we leave.

In [10]:
from contextlib import AsyncExitStack

from mcp import types


class MCPClient:
    def __init__(self, server_params, errlog=SERVER_LOG):
        self.server_params = server_params
        self.errlog = errlog
        self._session = None
        self._exit_stack = AsyncExitStack()

    async def connect(self):
        read_stream, write_stream = await self._exit_stack.enter_async_context(
            stdio_client(self.server_params, errlog=self.errlog))           # start the server process
        self._session = await self._exit_stack.enter_async_context(
            ClientSession(read_stream, write_stream))                        # open an MCP session
        await self._session.initialize()                                     # the handshake

    def session(self):
        if self._session is None:
            raise ConnectionError("Not connected. Use 'async with MCPClient(...)' or call connect().")
        return self._session

    # What the app needs for TOOLS
    async def list_tools(self) -> list[types.Tool]:
        return (await self.session().list_tools()).tools

    async def call_tool(self, tool_name: str, tool_input: dict) -> types.CallToolResult:
        return await self.session().call_tool(tool_name, tool_input)

    # Cleanup
    async def cleanup(self):
        await self._exit_stack.aclose()          # closes the session, then stops the server
        self._session = None

    async def __aenter__(self):
        await self.connect()
        return self

    async def __aexit__(self, *exc_info):
        await self.cleanup()


# Test the client on its own: we should see both tools.
async with MCPClient(SERVER_PARAMS) as mcp_client:
    listed = await mcp_client.list_tools()
    for tool in listed:
        print(f"{tool.name}: parameters {list(tool.input_schema['properties'])}")

assert len(listed) == 2

read_doc_contents: parameters ['doc_id']
edit_document: parameters ['doc_id', 'old_str', 'new_str']


**Connecting Claude to the server.** This is the full flow from Chapter 2. Two small adapters do the translation:

- **MCP tool to Claude tool.** Both use JSON Schema under `input_schema`, so it's a direct copy.
- **MCP result to Claude `tool_result`.** We join the result's text and copy `is_error`.

The loop is notebook 05's: call Claude, and while it asks for tools, run them **through the MCP client** and send the results back.

In [11]:
def to_claude_tools(mcp_tools):
    return [{"name": tool.name, "description": tool.description or "", "input_schema": tool.input_schema}
            for tool in mcp_tools]


async def run_with_mcp(mcp_client, messages, max_turns=8, max_tokens=2000):
    """The tool loop from notebook 05, with the tools coming from the MCP server."""
    tools = to_claude_tools(await mcp_client.list_tools())                 # step 2: ListTools
    for turn in range(1, max_turns + 1):
        response = track(client.messages.create(                           # step 3: ask Claude
            model=MODEL,
            max_tokens=max_tokens,
            output_config={"effort": "low"},
            messages=messages,
            tools=tools,
        ))
        messages.append({"role": "assistant", "content": response.content})
        tool_uses = [block for block in response.content if block.type == "tool_use"]
        print(f"Turn {turn}: stop_reason={response.stop_reason}, tools requested: {[t.name for t in tool_uses] or 'none'}")
        if response.stop_reason != "tool_use":
            print(f"\nAnswer:\n{text_from_message(response)}")
            return response

        results = []
        for tool_use in tool_uses:                                         # step 5: CallTool, through MCP
            mcp_result = await mcp_client.call_tool(tool_use.name, tool_use.input)
            text = " ".join(item.text for item in mcp_result.content if item.type == "text")
            print(f"      {tool_use.name}({tool_use.input}) -> {'ERROR: ' if mcp_result.is_error else ''}{text[:80]!r}")
            results.append({"type": "tool_result", "tool_use_id": tool_use.id,
                            "content": text, "is_error": bool(mcp_result.is_error)})
        messages.append({"role": "user", "content": results})               # step 7: results to Claude

    raise RuntimeError(f"Stopped after {max_turns} turns without a final answer.")


async with MCPClient(SERVER_PARAMS) as mcp_client:
    final = await run_with_mcp(mcp_client, [{"role": "user", "content":
        "What does Halvora's cybersecurity document say about INC-2023-Q4-011? Two sentences."}])

assert final.stop_reason == "end_turn"

[input=689 output=59 cost=$0.00197 | spent $0.0029 of $0.20]
Turn 1: stop_reason=tool_use, tools requested: ['read_doc_contents']
      read_doc_contents({'doc_id': 'cybersecurity'}) -> ERROR: 'Error executing tool read_doc_contents: Doc with id cybersecurity not found. Ava'
[input=888 output=63 cost=$0.00241 | spent $0.0053 of $0.20]
Turn 2: stop_reason=tool_use, tools requested: ['read_doc_contents']
      read_doc_contents({'doc_id': 'cybersecurity-analysis'}) -> '## Section 10: Cybersecurity Analysis\n\n**Incident Response Report: INC-2023-Q4-0'
[input=1,332 output=213 cost=$0.00479 | spent $0.0101 of $0.20]
Turn 3: stop_reason=end_turn, tools requested: none

Answer:
INC-2023-Q4-011 was a severity 1 incident detected on 14 November 2023. An attacker exploited an unpatched authentication library on a legacy reporting service and had access to internal dashboards for about 31 hours. No evidence showed that customer data was copied out, but it couldn't be ruled out, so customers and 

**What to notice**

- **Same question as the baseline in Chapter 3, very different answer.** This time Claude asks for `read_doc_contents`, the server reads the document, and the answer is grounded in it.
- **Watch the `doc_id` Claude picks.** It only has the tool's description, so it may guess something like `cybersecurity`, get our `ToolError` listing the valid ids, and retry with `cybersecurity-analysis`. That's the error message doing its job.
- **Your app never touched a document.** It only listed tools and passed calls along. That's the MCP split: the server owns the integration, and the client just carries messages.

## Chapter 7 · Defining resources: data, not actions (free)

**Tools are actions:** Claude decides to *do* something. **Resources are data** that the **app** reads and puts straight into the prompt, like `GET` endpoints in a web API.

**The use case from the course: `@mentions`.** When the user types `@software-engineering`, the app should (1) **list** the documents for autocomplete, and (2) **fetch** the one they picked and add it to the prompt. No tool call needed.

Every resource has a **URI**, its address:

| Type | URI | Returns |
|---|---|---|
| **Direct** (fixed address) | `docs://documents` | The list of document ids, as JSON |
| **Templated** (has parameters) | `docs://documents/{doc_id}` | One document, as plain text |

For a templated resource, the SDK reads `{doc_id}` out of the URI and passes it to your function. The **MIME type** tells the client what format the data is in.

In [12]:
%%writefile halvora_mcp/resources.py
# resources.py: DATA the app can read and put into the prompt. (Chapter 7)
from mcp.server.mcpserver.exceptions import ResourceNotFoundError

from core import docs, mcp


@mcp.resource("docs://documents", mime_type="application/json")
def list_docs() -> list[str]:
    """All document ids, for @mention autocomplete."""
    return list(docs.keys())


@mcp.resource("docs://documents/{doc_id}", mime_type="text/plain")
def fetch_doc(doc_id: str) -> str:
    """One document's full text."""
    if doc_id not in docs:
        raise ResourceNotFoundError(f"Doc with id {doc_id} not found")
    return docs[doc_id]

Writing halvora_mcp/resources.py


In [13]:
async with stdio_client(SERVER_PARAMS, errlog=SERVER_LOG) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()
        direct = (await session.list_resources()).resources
        templates = (await session.list_resource_templates()).resource_templates

print("Direct resources:")
for resource in direct:
    print(f"   {resource.uri}  ({resource.mime_type})")
print("Resource templates:")
for template in templates:
    print(f"   {template.uri_template}  ({template.mime_type})")

assert [str(resource.uri) for resource in direct] == ["docs://documents"]
assert [template.uri_template for template in templates] == ["docs://documents/{doc_id}"]

Direct resources:
   docs://documents  (application/json)
Resource templates:
   docs://documents/{doc_id}  (text/plain)


**What to notice**

- `docs://documents` is listed under **resources**, and `docs://documents/{doc_id}` under **templates**. The Inspector shows them in two tabs for the same reason: a template needs a value filled in before it can be read.
- `list_docs()` returns a Python **list**, and the SDK turns it into JSON for us because of `application/json`. No `json.dumps` needed.
- **Each connection starts a fresh server process**, and `server.py` loads every chapter file that exists. So the new `resources.py` was picked up without any restart. If you edit it, just reconnect.

## Chapter 8 · Accessing resources: `@mentions` (paid)

The client gets one new method, **`read_resource(uri)`**. The response holds a `contents` list. Usually you need only `contents[0]`, which carries the data **and** its MIME type, so we can parse it correctly:

- `application/json` becomes a Python object, through `json.loads`
- anything else stays a string

Then the app logic: find each `@doc_id` in the user's message, fetch it as a **resource**, and put it in the prompt inside `<document>` tags. Claude can then answer **without a single tool call**.

In [14]:
import re


async def read_resource(self, uri: str):
    result = await self.session().read_resource(uri)     # SDK 2.x takes a plain string
    resource = result.contents[0]                        # the data plus its mime_type
    if isinstance(resource, types.TextResourceContents):
        if resource.mime_type == "application/json":
            return json.loads(resource.text)
        return resource.text
    return resource                                      # binary data: leave it as it is


MCPClient.read_resource = read_resource                  # add the new method to the class


async def expand_mentions(mcp_client, user_text):
    """Find @doc_id mentions, and build a prompt with those documents attached."""
    known = await mcp_client.read_resource("docs://documents")              # the direct resource
    mentioned = [m for m in re.findall(r"@([a-z0-9-]+)", user_text) if m in known]
    blocks = [f'<document id="{doc_id}">\n{await mcp_client.read_resource(f"docs://documents/{doc_id}")}\n</document>'
              for doc_id in mentioned]                                       # the templated resource
    return mentioned, "\n\n".join(blocks + [user_text])


async with MCPClient(SERVER_PARAMS) as mcp_client:
    doc_ids = await mcp_client.read_resource("docs://documents")
    print(f"Autocomplete list ({type(doc_ids).__name__}, parsed from JSON): {doc_ids}\n")

    question = "Using @software-engineering, how many bugs were fixed in 2023 and how did outages change?"
    mentioned, prompt = await expand_mentions(mcp_client, question)
    print(f"Mentioned: {mentioned}, so the prompt is {len(prompt):,} characters\n")
    final = await run_with_mcp(mcp_client, [{"role": "user", "content": prompt}])

assert isinstance(doc_ids, list) and "software-engineering" in doc_ids
assert mentioned == ["software-engineering"]

Autocomplete list (list, parsed from JSON): ['executive-summary', 'table-of-contents', 'methodology', 'medical-research', 'software-engineering', 'financial-analysis', 'product-development', 'legal-developments', 'human-resources', 'marketing-and-brand', 'supply-chain-and-operations', 'sustainability', 'cybersecurity-analysis', 'research-partnerships', 'future-outlook']

Mentioned: ['software-engineering'], so the prompt is 1,363 characters

[input=1,087 output=81 cost=$0.00298 | spent $0.0131 of $0.20]
Turn 1: stop_reason=end_turn, tools requested: none

Answer:
In 2023, engineers fixed 1,284 bugs, up from 1,037 in 2022.

Customer-facing outages fell from 19 to 7. Average recovery time also improved, dropping from 52 minutes to 18 minutes.


**What to notice**

- `read_resource("docs://documents")` came back as a Python **list**, because the MIME type said JSON and we parsed it.
- **The answer likely arrives on turn 1, with no tools requested.** The document was already in the prompt, so no tool round trip was needed. Compare that with Chapter 6, which took at least two turns: when the user already knows which document matters, a resource is **faster and cheaper**.
- **Who decided?** In Chapter 6 **Claude** chose to read the document, through a tool. Here the **app** put it in the prompt because the **user** mentioned it, through a resource. That's the difference between the two primitives.
- `MCPClient.read_resource = read_resource` adds a method to a class after it was defined. That's fine in a notebook. In a real `.py` file, you'd write it inside the class.

## Chapter 9 · Defining prompts: the server's tested recipes (free)

**Analogy:** a restaurant's signature recipe. You *could* write "convert this to markdown" yourself, but the chef's tested recipe gives a better, more consistent result. **MCP prompts** let the server's author ship their best instructions.

A prompt is a function decorated with `@mcp.prompt(...)` that **returns a list of messages**, ready to send to Claude. Its arguments are filled into the template.

Two prompts:

- **`format`**, from the course: rewrite a document in Markdown, using the server's own `edit_document` tool. Prompts and tools working together.
- **`board_brief`**, specific to Halvora: three board-ready bullets with numbers. The kind of domain knowledge a server's author would package.

In [15]:
%%writefile halvora_mcp/prompts.py
# prompts.py: ready-made, tested INSTRUCTIONS the user can pick. (Chapter 9)
from mcp.server.mcpserver.prompts import base
from pydantic import Field

from core import mcp


@mcp.prompt(name="format", description="Rewrites the contents of a document in Markdown format.")
def format_document(
    doc_id: str = Field(description="Id of the document to format"),
) -> list[base.Message]:
    prompt = f"""Your goal is to reformat a document to be written with markdown syntax.

The id of the document you need to reformat is:
<document_id>
{doc_id}
</document_id>

Add headers, bullet points, tables, etc. as needed. Feel free to add in structure.
Use the 'read_doc_contents' tool to read the document first. Then use the 'edit_document' tool
to replace the WHOLE current text with the reformatted version in one edit.
Do not change any facts or numbers. Reply with a one-line summary when done."""
    return [base.UserMessage(prompt)]


@mcp.prompt(name="board_brief", description="Summarises a document as 3 board-ready bullet points.")
def board_brief(
    doc_id: str = Field(description="Id of the document to summarise"),
) -> list[base.Message]:
    prompt = f"""You are briefing Halvora's board of directors.

Read the document with id '{doc_id}' using the 'read_doc_contents' tool.
Then write exactly 3 bullet points:
- each one sentence, at most 25 words,
- each containing at least one number from the document,
- ordered from most to least important for the board.
No introduction, no conclusion, only the 3 bullets."""
    return [base.UserMessage(prompt)]

Writing halvora_mcp/prompts.py


In [16]:
async with stdio_client(SERVER_PARAMS, errlog=SERVER_LOG) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()
        prompts = (await session.list_prompts()).prompts
        for prompt in prompts:
            print(f"{prompt.name}: {prompt.description} | arguments: {[a.name for a in prompt.arguments or []]}")

        filled = await session.get_prompt("board_brief", {"doc_id": "financial-analysis"})

print("\nget_prompt('board_brief', doc_id='financial-analysis') returns:")
for message in filled.messages:
    print(f"   role={message.role}\n   " + message.content.text.replace("\n", "\n   "))

assert {prompt.name for prompt in prompts} == {"format", "board_brief"}
assert "financial-analysis" in filled.messages[0].content.text

format: Rewrites the contents of a document in Markdown format. | arguments: ['doc_id']
board_brief: Summarises a document as 3 board-ready bullet points. | arguments: ['doc_id']

get_prompt('board_brief', doc_id='financial-analysis') returns:
   role=user
   You are briefing Halvora's board of directors.
   
   Read the document with id 'financial-analysis' using the 'read_doc_contents' tool.
   Then write exactly 3 bullet points:
   - each one sentence, at most 25 words,
   - each containing at least one number from the document,
   - ordered from most to least important for the board.
   No introduction, no conclusion, only the 3 bullets.


**What to notice**

- `list_prompts()` returns each prompt's **name, description and arguments**. That's what a client app shows the user as a `/command` menu.
- `get_prompt(...)` returns **messages with the argument already filled in**: `financial-analysis` is in the text. The server did the templating, so the client never needs to know the template.
- Both prompts **tell Claude to use the server's own tools**. A good MCP server designs its tools, resources and prompts to work together.

## Chapter 10 · Prompts in the client: `/commands` (paid)

The last two client methods:

- **`list_prompts()`**: every prompt the server offers.
- **`get_prompt(name, args)`**: one prompt with its arguments filled in, as messages.

MCP prompt messages and Claude messages look almost the same: a `role` plus text. The cell below converts them, then runs the usual tool loop, because the prompts ask Claude to call tools.

In [17]:
async def list_prompts(self) -> list[types.Prompt]:
    return (await self.session().list_prompts()).prompts


async def get_prompt(self, prompt_name: str, args: dict[str, str]):
    return (await self.session().get_prompt(prompt_name, args)).messages


MCPClient.list_prompts = list_prompts
MCPClient.get_prompt = get_prompt


def to_claude_messages(prompt_messages):
    return [{"role": message.role, "content": message.content.text} for message in prompt_messages]


async with MCPClient(SERVER_PARAMS) as mcp_client:
    commands = [f"/{prompt.name}" for prompt in await mcp_client.list_prompts()]
    print("Available /commands:", commands, "\n")
    messages = to_claude_messages(await mcp_client.get_prompt("board_brief", {"doc_id": "financial-analysis"}))
    final = await run_with_mcp(mcp_client, messages)

assert set(commands) == {"/format", "/board_brief"}
assert final.stop_reason == "end_turn"

Available /commands: ['/format', '/board_brief'] 

[input=779 output=59 cost=$0.00215 | spent $0.0152 of $0.20]
Turn 1: stop_reason=tool_use, tools requested: ['read_doc_contents']
      read_doc_contents({'doc_id': 'financial-analysis'}) -> '## Section 3: Financial Analysis\n\nTotal revenue for 2023 was $412 million, an in'
[input=1,201 output=130 cost=$0.00370 | spent $0.0189 of $0.20]
Turn 2: stop_reason=end_turn, tools requested: none

Answer:
- Total 2023 revenue reached $412 million, up 14% from 2022, and subscriptions made up 81% of it.
- Year-end cash was $96 million with no long-term debt, which supports the board-approved $25 million acquisition budget for 2024.
- The Q4 security event cost an estimated $3.8 million, and we expect another $1.2 million in 2024 for security improvements.


**What to notice**

- **Three primitives, three triggers:** the **user** picked the prompt (`/board_brief`), the prompt told **Claude** to use a **tool**, and the app ran it through MCP.
- The brief's format (3 bullets, numbers, at most 25 words) came from the **server's** prompt, not from anything typed in this notebook. Any app using this server gets the same quality.

### Finale · a mini chat app

The course builds a command-line chat app. Here's the notebook version: **one MCP session**, and a `chat()` function that understands:

- `/command doc_id`, which runs a server **prompt**,
- `@doc_id`, which attaches a document as a **resource**,
- anything else, which is plain chat with the server's **tools** available.

Because everything happens in **one session**, the `/format` edit is still there when we read the document back at the end.

In [18]:
async def chat(mcp_client, user_input):
    print(f"\nYou: {user_input}")
    if user_input.startswith("/"):                                   # /command: a PROMPT
        name, _, doc_id = user_input[1:].partition(" ")
        messages = to_claude_messages(await mcp_client.get_prompt(name, {"doc_id": doc_id.strip()}))
        print(f"   (/{name}: a server prompt with {len(messages)} message(s))")
    else:                                                            # @mentions: RESOURCES
        mentioned, prompt = await expand_mentions(mcp_client, user_input)
        if mentioned:
            print(f"   (@mentions attached as resources: {mentioned})")
        messages = [{"role": "user", "content": prompt}]
    return await run_with_mcp(mcp_client, messages, max_tokens=4000)  # TOOLS are available throughout


async with MCPClient(SERVER_PARAMS) as mcp_client:
    print("Session open. Documents:", await mcp_client.read_resource("docs://documents"))
    original = await mcp_client.read_resource("docs://documents/research-partnerships")
    await chat(mcp_client, "@research-partnerships How many partnerships ended, and why? One sentence.")
    await chat(mcp_client, "/format research-partnerships")
    after_format = await mcp_client.read_resource("docs://documents/research-partnerships")

print("\n--- research-partnerships, read back as a resource in the SAME session ---")
print(after_format)
print("\nChanged by /format:", after_format != original)

Session open. Documents: ['executive-summary', 'table-of-contents', 'methodology', 'medical-research', 'software-engineering', 'financial-analysis', 'product-development', 'legal-developments', 'human-resources', 'marketing-and-brand', 'supply-chain-and-operations', 'sustainability', 'cybersecurity-analysis', 'research-partnerships', 'future-outlook']

You: @research-partnerships How many partnerships ended, and why? One sentence.
   (@mentions attached as resources: ['research-partnerships'])
[input=871 output=21 cost=$0.00195 | spent $0.0209 of $0.20]
Turn 1: stop_reason=end_turn, tools requested: none

Answer:
Two partnerships ended during the year because their projects were completed.

You: /format research-partnerships
   (/format: a server prompt with 1 message(s))
[input=826 output=59 cost=$0.00224 | spent $0.0231 of $0.20]
Turn 1: stop_reason=tool_use, tools requested: ['read_doc_contents']
      read_doc_contents({'doc_id': 'research-partnerships'}) -> '## Section 11: Researc

**What to notice**

- **`@research-partnerships`** put the document into the prompt as a **resource**, and Claude likely answered in one turn.
- **`/format research-partnerships`** ran the server's **prompt**, which told Claude to read the document and rewrite it with `edit_document`. That takes several **tool** turns.
- **The read-back should show the Markdown version**, because it's the same session, so the same server process and the same in-memory `docs`. Close the session, and the original text comes back, as Chapter 5 showed.

## Final spend

In [19]:
print(f"This notebook spent ${spent:.4f} of its ${BUDGET_USD:.2f} budget.")

This notebook spent $0.0340 of its $0.20 budget.


## Takeaways

| | Tools | Resources | Prompts |
|---|---|---|---|
| **What** | Actions | Data for the prompt | Tested instructions |
| **Triggered by** | **Claude** | **The app**, e.g. an `@mention` | **The user**, e.g. a `/command` |
| **Server side** | `@mcp.tool` | `@mcp.resource(uri)` | `@mcp.prompt` |
| **Client side** | `list_tools`, `call_tool` | `list_resources`, `read_resource` | `list_prompts`, `get_prompt` |

1. **MCP doesn't replace tool use; it moves the tools** into reusable servers. Your app lists them and passes calls along.
2. **The SDK writes the protocol for you:** decorators and type hints on the server, `ClientSession` on the client.
3. **Use `ToolError` for expected failures**, so Claude sees the message and can recover.
4. **Test the server on its own first,** with the Inspector or a chain check you can verify, like read, edit, read.
5. **stdio servers are child processes.** Each connection starts a fresh one, and in-memory state dies with it.

## Your turn

Each paid exercise costs about a cent or two.

- **Free:** add a `search_documents(query: str)` tool that returns the ids of documents containing a word, and list the tools again in Chapter 4.
- Then ask Claude *"Which documents mention XDR-47?"* using the loop from Chapter 6.
- In Chapter 8, mention two documents at once (`@financial-analysis` and `@future-outlook`). How much bigger does the prompt get?
- In Chapter 9, write a `compare` prompt with **two** arguments (`doc_a`, `doc_b`), and call it from `chat()`.
- **Free:** run `mcp dev halvora_mcp/server.py` in a terminal, and repeat the read, edit, read check in the Inspector.
- **Connect the same server to Claude Code**, and ask about Halvora there: the same server, a different client. [Notebook 12](12_claude_code_and_computer_use.ipynb) shows how.

**Next:** [11 · Agents and workflows](11_agents_and_workflows.ipynb): patterns for tasks that need more than one call to Claude.